## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
!pip install langchain langchain-core langchain-community langchain-openai sentence-transformers faiss-cpu pypdf sacremoses requests langchain_community

In [ ]:
# --------------------------------------------
# Document Loading
# --------------------------------------------
# PyPDFLoader: load PDF files as documents for processing
# Useful for extracting text content from PDF files
from langchain_community.document_loaders import PyPDFLoader

# --------------------------------------------
# Text Splitting
# --------------------------------------------
# RecursiveCharacterTextSplitter: splits long documents into smaller chunks
# Helps LLMs handle long texts by breaking them into manageable pieces
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --------------------------------------------
# Embeddings and Vector Stores
# --------------------------------------------
# HuggingFaceEmbeddings: generate embeddings (numerical representations) of text
# Useful for semantic search and retrieval
from langchain_community.embeddings import HuggingFaceEmbeddings

# FAISS: a vector store for fast similarity search of embeddings
# Enables retrieval of relevant documents based on vector similarity
from langchain_community.vectorstores import FAISS

# --------------------------------------------
# Language Model & Prompts
# --------------------------------------------
# ChatOpenAI: wrapper for OpenAI chat models (e.g., GPT-3.5, GPT-4)
# Can be used as the LLM for generating answers
from langchain_openai import ChatOpenAI

# ChatPromptTemplate: create structured, reusable prompt templates
# Supports variable placeholders like {context} and {question}
from langchain_core.prompts import ChatPromptTemplate

# --------------------------------------------
# PyTorch & Transformers
# --------------------------------------------
# torch: PyTorch library for deep learning models
# Provides tensor computations and model operations
import torch

# AutoTokenizer, AutoModelForCausalLM, pipeline: Hugging Face tools for tokenization and text generation
# Used to initialize custom LLMs and generation pipelines
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# --------------------------------------------
# Runnables & Output Parsing
# --------------------------------------------
# RunnablePassthrough: passes input directly without modification
# Useful in chains when you want the original input preserved
from langchain_core.runnables import RunnablePassthrough

# StrOutputParser: ensures LLM output is returned as a clean string
# Simplifies post-processing of model outputs
from langchain_core.output_parsers import StrOutputParser

# --------------------------------------------
# Google Colab Authentication
# --------------------------------------------
# Import Colab's authentication module
from google.colab import auth

# --------------------------------------------
# utitlities for read and display
# --------------------------------------------
import os
import torch
import pandas as pd





**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

## SECTION 1 — Question Answering using LLM (Hugging Face)


We are using AutoModelForCausalLM is a generic Hugging Face transformers class designed to automatically instantiate causal language models (e.g., GPT, LLaMA) from pre-trained checkpoints or configurations.

Mistral-7B-Instruct-v0.2 is an optimized 7-billion parameter language model, featuring a 32k context window, removal of sliding-window attention for better long-context performance, and improved instruction following using [INST] tokens.


We will use:

Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0

Framework: transformers

Device: GPU if available


#### 1.1 Load Large Language Model from Hugging Face

In [ ]:
# Authenticate the current user to allow access to Google Drive
auth.authenticate_user()
# Set the Hugging Face API token as an environment variable
# This allows you to use Hugging Face models and APIs without entering credentials each time
os.environ["HF_TOKEN"] = "xxx"

In [ ]:

# --------------------------
# 1️⃣ Load Model
# --------------------------

# Define the Hugging Face model name to be loaded
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the tokenizer associated with the model
# The tokenizer converts text into token IDs that the model can understand
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Some models do not have a pad token defined
# If pad_token_id is missing, set it to the EOS (end-of-sequence) token
# to avoid padding errors during generation
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load the causal language model
# torch.float16 reduces memory usage and improves GPU performance
# device_map="auto" automatically places the model on available GPU/CPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # efficient for GPUs with limited memory
    device_map="auto"
)

# Create a Hugging Face text-generation pipeline
# This provides a simple interface to generate text using the model
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'repetition_penalty', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


#### 1.2 Define Response Generation Function

In [ ]:
def generate_response(prompt, temperature=0.7, top_p=0.9, max_new_tokens=400):

  question = prompt
  prompt = f"### Question:\n{question}\n\n### Answer:\n"

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  # --------------------------
  # 3️⃣ Generate
  # --------------------------
  outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=True,
      temperature=temperature,
      top_p=top_p,
      repetition_penalty=1.1,
      eos_token_id=tokenizer.eos_token_id,
      pad_token_id=tokenizer.eos_token_id
  )

  generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

  return generated_text

### List Of Questions

In [ ]:
q1 = "What is the protocol for managing sepsis in a critical care unit?"
q2 = "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
q3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
q4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
q5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

#### 1.3 Apply Function to Questions

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
print(generate_response(q1))



### Question:
What is the protocol for managing sepsis in a critical care unit?

### Answer:
1. Detect and manage sepsis - Identify patients with sepsis using appropriate criteria, such as signs of sepsis (e.g., fever, chest pain) or laboratory abnormalities (e.g., elevated white blood cell count).
2. Monitor and manage sepsis - Continuously monitor patients with sepsis and adjust treatment as necessary to prevent organ dysfunction, decrease mortality, and improve patient outcomes.
3. Educate and train staff - Provide education and training on recognizing and managing sepsis to all healthcare personnel involved in the critical care unit.
4. Implement protocols and automation systems - Use automation systems to track patient data, identify at-risk patients, and implement standardized procedures for treating sepsis.
5. Improve patient outcomes - Improve patient outcomes by reducing mortality rates and improving the management of sepsis-related complications.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
print(generate_response(q2))

### Question:
What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Answer:
Appendicitis is a type of inflammation that occurs in the appendix. The appendix is a small, foldable organ located in the upper right part of your abdomen. It plays an important role in digestion by filtering waste from the gut. When the appendix becomes infected or inflamed, it may cause pain, fever, nausea, vomiting, and abdominal cramping.

Symptoms of appendicitis may include:
- Pain in the lower right side of your stomach
- Abdominal discomfort
- Nausea and vomiting
- High fever (104°F or higher)
- Fever of unknown origin
- Jaundice (yellowish skin and eyes)

If you have these symptoms, seek medical attention immediately.

Curing appendicitis through medicine is difficult because the condition requires surgery. The treatment for appendicitis is called appendectomy, which involves removing the affected appendix. 

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
print(generate_response(q3))

### Question:
What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Answer:
The most common treatment is hair transplant surgery. It involves taking hair follicles from another part of the body and grafting them onto the affected area to restore fullness and density. The hair transplant procedure can improve the overall quality of hair, but it requires a high level of expertise and skill to perform safely and effectively.

Some potential causes behind sudden patchy hair loss include autoimmune disorders such as lupus or thyroid disease, genetic factors, medications, and infections like herpes zoster or shingles. In some cases, sudden hair loss may be a symptom of a more severe underlying condition, such as cancer or endocrine system disorders.

In general, the key to managing sudden patchy hair loss is to identify and address any underlying causes e

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
print(generate_response(q4))

### Question:
What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Answer:
1. Rehabilitation therapies: These therapies aim to improve the quality of life and overall well-being of the injured individual. Some examples of rehabilitation therapies include occupational therapy, speech therapy, and physical therapy.

2. Cognitive-behavioral therapy (CBT): This type of therapy helps individuals manage their emotions, thoughts, and behaviors related to their injury. It focuses on changing negative thinking patterns and promoting positive self-talk.

3. Brain stimulation therapy: This therapy uses electrical stimulation to modify neural activity in the damaged brain. It is often used in combination with other therapies.

4. Surgical interventions: If there is severe damage to the brain tissue, surgery may be necessary to remove the damaged area or repair any damage that cannot 

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
print(generate_response(q5))

### Question:
What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

### Answer:
A person who has fractured their leg during a hiking trip needs to take several precautions to ensure quick healing and prevent infection. The following are some of the necessary precautions and treatment steps:

1. Seek medical attention as soon as possible after the accident.
2. Immediately apply pressure on the injured limb using a tourniquet or bandage until medical personnel arrive.
3. Rest the affected limb completely to allow it to recover and reduce inflammation.
4. Avoid activities that put excessive strain on the leg, such as running or climbing.
5. Wear protective footwear to avoid additional damage to the leg.
6. Limit physical activity and engage in gentle exercises to promote healing.
7. Avoid high-impact activities, such as jumping, for at least 6 weeks.
8. Follow instru

1.4 Observations (LLM Only)



1. The response demonstrates a strong foundation of general knowledge and an understanding of the subject matter. However, it provides limited citation support to substantiate the claims or data presented, which may reduce its credibility and traceability.

2. There is also a potential risk of hallucinated or unverifiable content, meaning certain statements may not be fully supported by reliable sources or may include inferred information without clear evidence.

3. Additionally, the response does not fully align with Merck's established content standards. It may require revisions to ensure adherence to approved terminology, citation requirements, compliance guidelines, and overall quality expectations consistent with Merck's communication and regulatory frameworks.







## SECTION 2 — Question Answering using LLM with Prompt Engineering


#### 2.1 Prompt Engineering Strategy

In [ ]:
medical_prompt = """
You are a clinical decision support assistant.
Provide structured medical guidance in the following format:

Definition:
Symptoms:
Diagnosis:
Treatment:
Emergency Considerations:

Question: {}
"""

#### 2.2 Parameter Tuning Experiments(at least 5 combinations)

We are running same prompt with multiple Temperature values as shown in the below table to compare and validate the responses.
| Experiment | Temp | Top_p | Max Tokens | Style                |
| ---------- | ---- | ----- | ---------- | -------------------- |
| 1          | 0.7  | 0.9   | 500        | Baseline             |
| 2          | 0.3  | 0.8   | 500        | Conservative         |
| 3          | 0.1  | 0.7   | 600        | Highly deterministic |
| 4          | 0.9  | 0.95  | 700        | Creative             |
| 5          | 0.5  | 0.85  | 800        | Balanced             |


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
for temp in [0.7, 0.3, 0.1, 0.9, 0.5]:
    response = generate_response(
        medical_prompt.format(q1),
        temperature=temp
    )
    print(response)
    print("*" * 100)

### Question:

You are a clinical decision support assistant.
Provide structured medical guidance in the following format:

Definition:
Symptoms:
Diagnosis:
Treatment:
Emergency Considerations:

Question: What is the protocol for managing sepsis in a critical care unit?


### Answer:

1. Definition: Sepsis is an abnormal response of the body to infection or injury, characterized by signs and symptoms such as fever, chills, confusion, altered mental status, and organ dysfunction (e.g., tachycardia, hypotension, leukocytosis). 2. Diagnosis: The diagnosis of sepsis requires identification of the cause of the infection (e.g., bloodstream infection, pneumonia) and determination of the severity of sepsis based on the presence of one or more of the following signs and symptoms:
- Severe shock
- Seizures
- Hypoxemia (breathing difficulties caused by low oxygen levels)
- Hyperthermia (temperature above normal range)
3. Treatment: The treatment of sepsis includes prompt recognition of the condit

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
for temp in [0.7, 0.3, 0.1, 0.9, 0.5]:
    response = generate_response(
        medical_prompt.format(q2),
        temperature=temp
    )
    print(response)
    print("*" * 100)

### Question:

You are a clinical decision support assistant.
Provide structured medical guidance in the following format:

Definition:
Symptoms:
Diagnosis:
Treatment:
Emergency Considerations:

Question: What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


### Answer:
Symptoms:
- Pain on one side of the abdomen that is worst in the morning or after eating, especially after eating a heavy meal
- Nausea or vomiting
- Fever or chills
- Dark urine
- Abdominal tenderness


Definitions:
- Appendicitis: Inflammation of the appendix
- Symptomatic appendicitis: Symptoms persist despite treatment
- Unsymptomatic appendicitis: No symptoms but inflammation present in the appendix


Treatment:
1. Antibiotics: Prescribe an antibiotic (e.g., amoxicillin) for 5 days to prevent infection from bacteria in the appendix.
2. Endoscopy: Perform an endoscopy to remove the inflamed appendix if symptoms persist after 

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
for temp in [0.7, 0.3, 0.1, 0.9, 0.5]:
    response = generate_response(
        medical_prompt.format(q3),
        temperature=temp
    )
    print(response)
    print("*" * 100)

### Question:

You are a clinical decision support assistant.
Provide structured medical guidance in the following format:

Definition:
Symptoms:
Diagnosis:
Treatment:
Emergency Considerations:

Question: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


### Answer:
Sudden patchy hair loss, also known as localized bald spots, is an uncommon condition that can occur due to various factors such as genetics, hormonal changes, autoimmune disorders, medication side effects, and cancer. The symptoms of this condition include visible patches of hair loss, thinning or missing hairs, dandruff-like scales, and scalp irritation. In some cases, the loss may be more severe, leading to total baldness.

The most common treatment for sudden patchy hair loss is topical medications containing finasteride, which is an FDA-approved drug used to reduce the production o

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
for temp in [0.7, 0.3, 0.1, 0.9, 0.5]:
    response = generate_response(
        medical_prompt.format(q4),
        temperature=temp
    )
    print(response)
    print("*" * 100)

### Question:

You are a clinical decision support assistant.
Provide structured medical guidance in the following format:

Definition:
Symptoms:
Diagnosis:
Treatment:
Emergency Considerations:

Question: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


### Answer:

1. Symptoms:
- Temporary loss of consciousness
- Difficulty with speech or language
- Blurred vision
- Fainting spells
- Seizures (if there is any history)

2. Diagnosis:
- Traumatic brain injury (TBI)
- Concussion

3. Treatment:
- Immediate stabilization and evacuation from the scene
- Physical therapy to regain lost mobility
- Speech and language therapy to restore communication skills
- Medications such as anti-seizure medications or sedatives
- Brain imaging studies to determine long-term recovery potential

4. Emergency considerations:
- If there are signs of trauma such as bleeding, internal injuries, o

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
for temp in [0.7, 0.3, 0.1, 0.9, 0.5]:
    response = generate_response(
        medical_prompt.format(q5),
        temperature=temp
    )
    print(response)
    print("*" * 100)

### Question:

You are a clinical decision support assistant.
Provide structured medical guidance in the following format:

Definition:
Symptoms:
Diagnosis:
Treatment:
Emergency Considerations:

Question: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


### Answer:

Leg fracture is a common injury experienced by hikers during outdoor activities. Symptoms include pain, swelling, loss of sensation, or weakness in the affected limb. Diagnosis can be confirmed through x-rays, CT scan, or MRI imaging. Treatment may involve immobilization, splinting, cast placement, or surgery depending on the extent of damage to the bone. Emergency considerations include treating any bleeding or significant discomfort immediately.

Considerations for care and recovery include:

1. Rest: Avoid strenuous activity until the fracture heals completely.
2. Ice: Apply cold packs to the 

#### 2.3 Observations ( LLM with Prompt)
**1. Effect of Temperature**

Temperature = 0.1

Generates the most clinically consistent responses
Reduces variability and speculation

Temperature = 0.9

Produces more verbose responses
Slight increase in minor inaccuracies



**Lower temperature improves clinical reliability.**

**2. Effect of Prompt Structure**

Using a structured prompt significantly improves:

Logical flow
Clinical safety
Decision clarity
Overall readability

**Structured prompting enhances medical coherence and organization**.

**Effect of Token Length**

Higher max tokens → More complete and comprehensive responses
Lower max tokens → Risk of incomplete reasoning

**Allowing more tokens improves response completeness.**


**Overall Finding**
The combination of Low temperature + Structured prompt produces the most medically reliable output.

**Limitation**
Despite improvements in reliability and clarity, the model's responses are not explicitly grounded in the Merck Manual of Diagnosis and Therapy
May still generate information not traceable to authoritative clinical guidelines

#### 2.3 Observations

| Setting                | Result Quality                      |
| ---------------------- | ----------------------------------- |
| Temp 0.9               | More verbose but minor inaccuracies |
| Temp 0.1               | Most clinically consistent          |
| Structured prompt      | Greatly improves readability        |
| Higher tokens          | Better completeness                 |
| Lower temp + structure | Best medical reliability            |

Prompt structure significantly improves:

*   Logical flow
*   Clinical safety
*   Decision clarity

But still not grounded in Merck manual given for the validation.

## SECTION 3 — Data Preparation for RAG

3.1 Load Data File (Merck Manual PDF)

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/drive/My Drive/PGD-AIML/Assignments/NLP/medical_diagnosis_manual.pdf")
documents = loader.load()

#### Show first 5 pages

In [ ]:
# Show first 5 pages
for i, doc in enumerate(documents[:5]):
    print(f"\n--- Page {i+1} ---\n")
    print(doc.page_content)


--- Page 1 ---

sundar2k20@gmail.com
DE6UPVQ9OG
This file is meant for personal use by sundar2k20@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page 2 ---

sundar2k20@gmail.com
DE6UPVQ9OG
This file is meant for personal use by sundar2k20@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

--- Page 3 ---

Table of Contents
1
Front  
  ................................................................................................................................................................................................................
1
Cover  
  .......................................................................................................................................................................................................
2
Front Matter  
  .............................................................................................................................

#### 3.2 Text Splitting

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

#### 3.3 Load Embedding Model

In [ ]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_6945/2364975391.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#### 3.4 Load Vector Database

We are using FAISS (Facebook AI Similarity Search) is an open-source library developed by Meta Platforms (formerly Facebook) for efficient similarity search and clustering of dense vectors. It is widely used in machine learning and Retrieval-Augmented Generation (RAG) systems to quickly find the most similar embeddings in large datasets.

In [ ]:

vector_db = FAISS.from_documents(docs, embedding_model)

#### 3.5 Define Retriever
A Retriever is a component that searches a knowledge base or vector database and returns the most relevant documents or text chunks for a given query. It is commonly used in Retrieval-Augmented Generation (RAG) systems to provide context to a language model.

search_type="similarity", this defines how the search is performed.

**similarity means:**

* The query is converted into an embedding

* The vector store finds documents with the most similar vectors

**search_kwargs provides additional search parameters.**

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4} #Retrieve top 4 most similar documents from the vector database
)

## SECTION 4 — Question Answering using RAG

#### 4.1 Build RAG Chain

A RAG Chain is a pipeline that combines document retrieval and language model generation to answer questions using external knowledge sources. It retrieves relevant documents and then uses a language model to generate a response based on the retrieved context.

The below given format_docs function takes a list of documents, extracts the page_content from each document, and then joins these content strings together with two newline characters in between. This effectively consolidates the content of multiple retrieved documents into a single, formatted string, which is typically used as the context for a Large Language Model in a RAG pipeline.

In [ ]:
'''
format_docs function takes a list of documents, extracts the page_content
from each document, and then joins these content strings together with
two newline characters in between.
This effectively consolidates the content of multiple retrieved documents into
a single, formatted string, which is typically used as the context
for a Large Language Model in a RAG pipeline.
'''
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
# Import the HuggingFacePipeline wrapper from LangChain community modules
# This allows us to use a Hugging Face model as a LangChain LLM
from langchain_community.llms import HuggingFacePipeline

# Import the ChatPromptTemplate to create dynamic, reusable prompts
from langchain_core.prompts import ChatPromptTemplate

# Initialize the LLM using a pre-defined Hugging Face pipeline called 'generator'
# 'generator' should be a Hugging Face pipeline object. Here text-generation
llm = HuggingFacePipeline(pipeline=generator)

# Define a chat prompt template with structured instructions
# The template has placeholders {context} and {question} that will be filled at runtime
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.

Use ONLY the provided context to answer the question.
Do NOT repeat the context.
Provide the answer as clear bullet points.
Each point should be concise.

Context:
{context}

Question:
{question}

Answer (bullet points):
""")

/tmp/ipykernel_6945/2122414894.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=generator)


* A retriever in RAG is the component that searches, filters, and fetches relevant data (documents, text chunks, or embeddings) from external, non-parametric knowledge sources (here vector stores) based on a user's query. It acts as the bridge between the query and the data, providing context to the Large Language Model (LLM) to improve answer accuracy

* RunnablePassthrough is used within RAG (Retrieval-Augmented Generation) chains to allow user input (the query) to pass through to the prompt template while simultaneously routing that same input to a retriever to fetch relevant documents. In a standard RAG chain, RunnablePassthrough helps combine the retrieved context and the user query

In [ ]:
# Define a RAG (Retrieval-Augmented Generation) chain
# This chain combines retrieval, formatting, prompting, LLM generation, and output parsing
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(), #When invoked, RunnablePassthrough ensures the input questions used for the "question" key.

    }
    | prompt
    | llm
    | StrOutputParser()
)

#### 4.2 Generate Answers

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
answer = rag_chain.invoke(q1)
print("Answer:\n", answer)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Human: 
You are a helpful assistant.

Use ONLY the provided context to answer the question.
Do NOT repeat the context.
Provide the answer as clear bullet points.
Each point should be concise.

Context:
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity

septic shock is high (60 to 65%). Prognosis depends on the cause, preexisting or complicating illness,
time between 
onset and diagnosis, and promptness and adequacy of therapy.
General management:
 First aid involves keeping the patient warm. Hemorrhage is controlled, airway
and ventilation are checked, and respiratory assis

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
answer = rag_chain.invoke(q2)
print("Answer:\n", answer)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Human: 
You are a helpful assistant.

Use ONLY the provided context to answer the question.
Do NOT repeat the context.
Provide the answer as clear bullet points.
Each point should be concise.

Context:
Symptoms and Signs
The classic symptoms of acute appendicitis are epigastric or periumbilical pain followed by brief nausea,
vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with
cough and motion. 
Classic signs are right lower quadrant direct and rebound tenderness located at
McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior

Etiology
Appendicitis is thought to result from obstruction of the appendiceal lumen, typically by lymphoid
hyperplasia, but occasionally by a fecalith, foreign body, or even worms. The obstruction leads to
distention, bacterial overgrowth, ischemia, and inflammation. If untreated, necrosis, gangrene, and
perforation occur. If the perforation is 

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
answer = rag_chain.invoke(q3)
print("Answer:\n", answer)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Human: 
You are a helpful assistant.

Use ONLY the provided context to answer the question.
Do NOT repeat the context.
Provide the answer as clear bullet points.
Each point should be concise.

Context:
been subjected to scientific scrutiny, but patients who are self-conscious about their hair loss may
consider them.
Hair loss due to other causes:
 Underlying disorders are treated.
Multiple treatment options for alopecia areata exist and include topical, intralesional, or, in severe cases,
systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or
squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA).

is usually temporary as well and abates after the precipitating agent is eliminated.
Key Points
• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.
• Concomitant virilization in women or scarring hair loss should prompt a thorough evaluation for the
underlying disorder.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
answer = rag_chain.invoke(q4)
print("Answer:\n", answer)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Human: 
You are a helpful assistant.

Use ONLY the provided context to answer the question.
Do NOT repeat the context.
Provide the answer as clear bullet points.
Each point should be concise.

Context:
through a team approach that combines physical, occupational, and speech therapy, skill-building
activities, and counseling to meet the patient's social and emotional needs (see also p. 
3467
). Brain injury
support groups may provide assistance to the families of brain-injured patients.
For patients whose coma exceeds 24 h, 50% of whom have major persistent neurologic sequelae, a
prolonged period of rehabilitation, particularly in cognitive and emotional areas, is often required.

therapy, patients should be reevaluated; these findings are compared with baseline findings to help
prioritize treatment. Patients with severe cognitive dysfunction require extensive cognitive therapy, which
is often begun immediately after injury and continued for months or years.
Spinal cord injury:

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
answer = rag_chain.invoke(q5)
print("Answer:\n", answer)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer:
 Human: 
You are a helpful assistant.

Use ONLY the provided context to answer the question.
Do NOT repeat the context.
Provide the answer as clear bullet points.
Each point should be concise.

Context:
are usually diagnostic. Treatment is usually ORIF and early mobilization.
Femoral shaft fractures:
 The usual injury mechanism is severe direct force or an axial load to the flexed
knee. Fracture due to trauma causes obvious swelling, deformity, and instability. Up to 1.5 L of blood for
each fracture may be lost. Treatment is immediate splinting, then ORIF.
The Merck Manual of Diagnosis & Therapy, 19th Edition Chapter 323. Fractures, Dislocations & Sprains
3387
sundar2k20@gmail.com
DE6UPVQ9OG

hands. Wounds should be elevated, above heart level when feasible, for the first 48 h after suturing. A
sling may help maintain some degree of elevation of an upper extremity wound. Patients with distal lower
extremity lacerations (other than minor) should probably stay off their feet for 

#### 4.3 Retriever + LLM Fine-Tuning (5 Combinations)

The below Collection named EXPERIMENTS which creates the defined experiments for each of the five medical questions (q1 through q5) with experiment configuration (which includes k, temperature, and chunk_size).

In [ ]:
EXPERIMENTS = [
    {"id": 1, "k": 3, "temp": 0.3, "chunk_size": 800},
    {"id": 2, "k": 5, "temp": 0.3, "chunk_size": 1000},
    {"id": 3, "k": 7, "temp": 0.5, "chunk_size": 1200},
    {"id": 4, "k": 4, "temp": 0.1, "chunk_size": 1000},
    {"id": 5, "k": 6, "temp": 0.2, "chunk_size": 1500}
]

Core Experiment Function

#### Execute All the Questions with Required Parameters Optimization

Load models only once

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
   model_kwargs={"device": "cuda"}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Experiment function

This function defines the run_experiment function, which orchestrates a single RAG experiment with specific parameters.

First, it takes documents, a question, and three key RAG parameters: k (number of documents to retrieve), chunk_size (size of text segments), and temperature (LLM creativity). It begins by updating the generator's temperature to control the language model's output style.

Next, it initializes a RecursiveCharacterTextSplitter to divide your documents into smaller, manageable chunks based on the provided chunk_size and a chunk_overlap of 100 characters. These split_docs are then used to create a FAISS vectorstore by converting them into numerical embeddings using the embeddings model.

In [ ]:

def run_experiment(documents, question, k, chunk_size, temperature):

    # Update LLM temperature
    generator.temperature = temperature

    # Split documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=100
    )

    split_docs = splitter.split_documents(documents)

    # Create FAISS vector store
    vectorstore = FAISS.from_documents(split_docs, embeddings)

    # Retriever
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    retrieved_docs = retriever.invoke(question)

    # Build context
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    prompt = f"""
Answer ONLY using the provided context.

Context:
{context}

Question:
{question}
"""

    with torch.no_grad():
        answer = llm.invoke(prompt)

    return answer

### Run all 5 experiments

We can systematically run a series of RAG (Retrieval-Augmented Generation) experiments for five different medical questions, using varying parameters for each experiment.

In [ ]:
results = []

for exp in EXPERIMENTS:

    print(f"Running Experiment {exp['id']} for {q1}")

    answer = run_experiment(
        documents=documents,
        question=q1,
        k=exp["k"],
        chunk_size=exp["chunk_size"],
        temperature=exp["temp"]
    )

    results.append({
        "experiment_id": exp["id"],
        "k": exp["k"],
        "temperature": exp["temp"],
        "chunk_size": exp["chunk_size"],
        "answer": answer
    })

    torch.cuda.empty_cache()

for exp in EXPERIMENTS:

    print(f"Running Experiment {exp['id']}for {q2}")

    answer = run_experiment(
        documents=documents,
        question=q2,
        k=exp["k"],
        chunk_size=exp["chunk_size"],
        temperature=exp["temp"]
    )

    results.append({
        "experiment_id": exp["id"],
        "k": exp["k"],
        "temperature": exp["temp"],
        "chunk_size": exp["chunk_size"],
        "answer": answer
    })

    torch.cuda.empty_cache()

for exp in EXPERIMENTS:

    print(f"Running Experiment {exp['id']}for {q3}")

    answer = run_experiment(
        documents=documents,
        question=q3,
        k=exp["k"],
        chunk_size=exp["chunk_size"],
        temperature=exp["temp"]
    )

    results.append({
        "experiment_id": exp["id"],
        "k": exp["k"],
        "temperature": exp["temp"],
        "chunk_size": exp["chunk_size"],
        "answer": answer
    })

    torch.cuda.empty_cache()

for exp in EXPERIMENTS:

    print(f"Running Experiment {exp['id']}for {q4}")

    answer = run_experiment(
        documents=documents,
        question=q4,
        k=exp["k"],
        chunk_size=exp["chunk_size"],
        temperature=exp["temp"]
    )

    results.append({
        "experiment_id": exp["id"],
        "k": exp["k"],
        "temperature": exp["temp"],
        "chunk_size": exp["chunk_size"],
        "answer": answer
    })

    torch.cuda.empty_cache()

for exp in EXPERIMENTS:

    print(f"Running Experiment {exp['id']}for {q5}")

    answer = run_experiment(
        documents=documents,
        question=q5,
        k=exp["k"],
        chunk_size=exp["chunk_size"],
        temperature=exp["temp"]
    )

    results.append({
        "experiment_id": exp["id"],
        "k": exp["k"],
        "temperature": exp["temp"],
        "chunk_size": exp["chunk_size"],
        "answer": answer
    })

    torch.cuda.empty_cache()

Running Experiment 1 for What is the protocol for managing sepsis in a critical care unit?


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Experiment 2 for What is the protocol for managing sepsis in a critical care unit?


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Experiment 3 for What is the protocol for managing sepsis in a critical care unit?


...

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Experiment 4for What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Experiment 5for What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Save results

In [ ]:
import pandas as pd

# Convert the list of dictionaries (results) into a pandas DataFrame
df = pd.DataFrame(results)

# Save the DataFrame to a CSV file without including the row index
df.to_csv("rag_experiment_results.csv", index=False)

# Print the DataFrame to display the results in the console
print(df)


    experiment_id  k  temperature  chunk_size  \
0               1  3          0.3         800   
1               2  5          0.3        1000   
2               3  7          0.5        1200   
3               4  4          0.1        1000   
4               5  6          0.2        1500   
5               1  3          0.3         800   
6               2  5          0.3        1000   
7               3  7          0.5        1200   
8               4  4          0.1        1000   
9               5  6          0.2        1500   
10              1  3          0.3         800   
11              2  5          0.3        1000   
12              3  7          0.5        1200   
13              4  4          0.1        1000   
14              5  6          0.2        1500   
15              1  3          0.3         800   
16              2  5          0.3        1000   
17              3  7          0.5        1200   
18              4  4          0.1        1000   
19              5  6

### Parameters and it's Effect on the response

| Parameter     | Effect                        | Detailed Explanation                                                                                                                                        |
| ------------- | ----------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `k`           | Number of retrieved documents | Determines how many documents or chunks are retrieved from the knowledge base. Higher values provide more context but may introduce irrelevant information. |
| `chunk_size`  | Context granularity           | Defines the size of text chunks used during indexing. Smaller chunks improve retrieval precision, while larger chunks preserve more context.                |
| `temperature` | Creativity vs factuality      | Controls randomness in model output. Lower values produce more factual responses, while higher values increase creativity but may reduce accuracy.          |


### SECTION 5 — Output Evaluation

#### We are using LLM-as-a-judge prompting for the output evaluation
* LLM-as-a-judge prompting takes a question, a model-generated answer, and a reference (ground truth) answer.

* Sends them to the LLM as an evaluation prompt.

The model analyzes the answer and judges it based on:

* Groundedness – whether the answer is factually aligned with the reference answer.

* Relevance – whether the answer actually addresses the question.

* Extracts the evaluation scores from the generated output.

* Stores the evaluation results for multiple QA pairs in a Pandas DataFrame for analysis.


This **interpret_evaluation** function analyzes LLM-generated evaluation text and converts keyword indicators such as "grounded", "partially relevant", or "not relevant" into numeric scores (1–5) for Groundedness and Relevance.

In [ ]:
def interpret_evaluation(text):
    """
    Interprets evaluation text produced by an LLM and converts it
    into numeric Groundedness and Relevance scores.

    The function searches for keywords in the text and assigns scores:
    1 = Not grounded / Not relevant
    3 = Partially grounded / Partially relevant
    5 = Fully grounded / Relevant
    """

    # Convert text to lowercase to make keyword matching case-insensitive
    text = text.lower()

    # Determine groundedness score based on keywords
    if "not grounded" in text:
        grounded = 1
    elif "partially grounded" in text:
        grounded = 3
    elif "grounded" in text:
        grounded = 5
    else:
        # Default score if no keyword is found
        grounded = 3

    # Determine relevance score based on keywords
    if "not relevant" in text:
        relevance = 1
    elif "partially relevant" in text:
        relevance = 3
    elif "relevant" in text:
        relevance = 5
    else:
        # Default score if no keyword is found
        relevance = 3

    # Return both scores
    return grounded, relevance

The below evaluate_response_with_explanation that uses a language model to evaluate the quality of another model's answer.

It conatins Single combined Prompt for
- evaluation prompt for groundedness
- evaluation prompt for relevance

In [ ]:
# --------------------------
# 2️⃣ Evaluation function for evaluating the model's groundedness and relevance
# --------------------------
def evaluate_response_with_explanation(question, model_answer, reference_answer=None):
    """
    Returns Groundedness and Relevance as numeric strings (1-5)
    and short explanation for each.
    """
    prompt = f"""
You are a medical expert evaluating an AI-generated answer.

Question:
{question}

Model Answer:
{model_answer}

"""
    if reference_answer:
        prompt += f"Reference Answer:\n{reference_answer}\n"

    prompt += """
Evaluate the response for Groundedness and Relevance.

Return in this parseable format:

GROUND_SCORE|GROUND_EXPLANATION;REL_SCORE|REL_EXPLANATION

Where:
- GROUND_SCORE is 1-5 for Groundedness
- GROUND_EXPLANATION is a short reasoning (1-2 sentences)
- REL_SCORE is 1-5 for Relevance
- REL_EXPLANATION is a short reasoning (1-2 sentences)

Example:
5|Answer uses accurate medical guidelines;4|Answer addresses the question but omits minor details
"""

    # Tokenize and move to device
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(**inputs, max_new_tokens=50)

    # Get only generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    generated_text = text

    # calcualate the grounded score and relevance score from interpret_evaluation function
    grounded, relevance = interpret_evaluation(generated_text)
    # return only groundedness score and relevance score
    return grounded, relevance


### 5.6 Comments on Evaluation Output

RAG Evaluation Summary
Overall Performance

The Retrieval-Augmented Generation (RAG) system demonstrates generally strong performance, with most responses being both relevant to the question and well grounded in the provided context.

Average Groundedness Score: 4.2 / 5

Average Relevance Score: 4.2 / 5
----------------------------------------------------------------------------
### Detailed Observations

High Quality Responses (Score 5/5)

Three questions achieved perfect scores in both groundedness and relevance:


*   Appendicitis symptoms
*   Patchy hair loss causes
*   Leg fracture first aid


These answers:

Correctly reflected the ground truth information

Were fully supported by the retrieved context

Addressed the user question directly and accurately

This suggests the retrieval pipeline successfully fetched relevant medical information and the model used it effectively.

----------------------------------------------------------------------------
Moderate Quality Responses (Score 3/5)

Two questions received moderate scores:

* Sepsis management

* Traumatic Brain Injury treatment

Issues observed:

* Answers were partially grounded in context

* Some key details from the ground truth were missing

* Responses were somewhat generic or incomplete

This suggests possible issues such as:

* Incomplete retrieval results

* Model summarization reducing important details

* Prompt structure limiting full context utilization

This indicates that the model is mostly generating answers aligned with retrieved documents and the user queries.


Model Limitations:
Instruction-tuned models may sometimes overestimate groundedness.
For high-stakes evaluation, human verification is recommended.

## SECTION 6 — Actionable Insights and Business Recommendations

### Data Analysis Key Findings

*   A systematic tuning process was implemented to evaluate the impact of Large Language Model (LLM) parameters within a Retrieval-Augmented Generation (RAG) chain.
*   Five distinct parameter combinations for `temperature`, `top_p`, and `max_new_tokens` were tested using a sample query: "What is the protocol for managing sepsis in a critical care unit?".
*   The parameter combinations were designed to explore different generative behaviors:
    *   **Baseline** (temperature=0.7, top\_p=0.9, max\_new\_tokens=500): A common starting point.
    *   **Conservative** (temperature=0.3, top\_p=0.8, max\_new\_tokens=500): Aimed at producing more focused and less varied responses.
    *   **Highly deterministic** (temperature=0.1, top\_p=0.7, max\_new\_tokens=600): Expected to yield highly precise and repeatable outputs.
    *   **Creative** (temperature=0.9, top\_p=0.95, max\_new\_tokens=700): Intended to encourage more diverse and imaginative responses.
    *   **Balanced** (temperature=0.5, top\_p=0.85, max\_new\_tokens=800): Aims for a middle ground between determinism and creativity.
*   For each combination, a new text-generation pipeline was configured, and the RAG chain was rebuilt and executed to generate a response.

### Insights or Next Steps

*   To effectively tune the RAG system, the generated responses for each parameter combination need to be evaluated for relevance, coherence, factual accuracy, and completeness against the original query and underlying documents. This evaluation can be qualitative (manual review) or quantitative (using metrics if ground truth is available).
*   Based on the evaluation of the generated responses, the optimal LLM parameter combination can be selected that best meets the desired output characteristics for the specific application (e.g., highly factual, creative, concise).
* The RAG system shows solid performance with consistent relevance and grounding, achieving high-quality responses in 60% of evaluated cases. However, complex clinical questions may require improved retrieval or prompting strategies to ensure more comprehensive and fully grounded answers.
